# 08 · Your first eval: a golden set and a test that needs no model
Level L2 · Part 2 · core · ~25 min · builds on 06

## Goal
Score 30 golden support tasks by their **end state**, with a pytest file that makes **no model call**. Read the task
success rate, and name each task the agent misses and why.

## Why it matters
Without an eval, you change a prompt, try three questions by hand, see nothing wrong, and ship. A rule that now
refunds "I changed my mind" slips through. The first person to notice is a customer, and the bill has already gone out.

| Without an eval | With a golden set |
|---|---|
| You try three questions by hand after each change | 30 tasks run in seconds after each change |
| You judge the wording of the reply | You check the end state: the refund exists, or it does not |
| A regression is found by a customer | A regression fails the test before the change ships |

**What it makes easy:** a score you can compare between two versions, and a list of the exact tasks that broke.
**When you don't need it:** a demo you will throw away. Anything a customer uses needs one.

## The idea
A **golden set** is a list of tasks where you already know the right answer: the customer's request and the
**expected end state**, meaning what the world must look like afterwards. A **grader** runs the agent and compares.

**Outcome grading** checks the world: a refund row exists, a ticket is open. **Trajectory grading** checks the path:
no forbidden tool ran, and the required lookup did run. A **task success rate** is passed tasks divided by all tasks.

The agent here is a **scripted stand-in**: fixed rules, no model. Lesson 06's loop uses a real model; this test swaps
it for rules, so it is free and repeatable. A real model runs against the same golden set in lesson 22.

Analogy: a driving test. The examiner checks that the car reached the address (outcome) and that the driver did not
cross a solid line (trajectory).

## Picture
```mermaid
flowchart LR
  G[Golden task: request + expected end state] --> A[Agent under test]
  A --> W[World after the run: refunds, tickets]
  A --> T[Tool calls made]
  W --> O{Outcome graded}
  T --> R{Trajectory graded}
  O --> S[Task success rate]
  R --> S
```

## Step by step
**Step 1 · the golden set (a peek).** The file is written by hand for this course and comes with the lesson. It holds
30 tasks and a world of 12 fictional orders. No personal data. The 30 tasks are provided: you do not write them here;
in Try it you write the 31st. The file also has 20 injection cases; ignore them for now, later lessons use them.
The printout shows one whole task: the request, the `expected_end`, and the two optional lists.

In [1]:
import json
import sys
from pathlib import Path

RUN = Path("../data/08")  # this lesson's run-time files
RUN.mkdir(parents=True, exist_ok=True)
GOLDEN = json.loads((Path("../data") / "support_eval.json").read_text())
print("orders:", len(GOLDEN["world"]["orders"]), "| tasks:", len(GOLDEN["tasks"]), "| injection cases:", len(GOLDEN["injection"]))
print(json.dumps(GOLDEN["tasks"][0], indent=1))  # t01: no refund, no ticket; the lookup must run, refund and ticket are forbidden

orders: 12 | tasks: 30 | injection cases: 20
{
 "id": "t01",
 "request": "Where is my parcel for order 1002? It was due on Monday.",
 "expected_end": {
  "refunds": {},
  "tickets": []
 },
 "must_call": [
  "lookup_order"
 ],
 "forbidden_tools": [
  "issue_refund",
  "open_ticket"
 ]
}


**Step 2 · the scripted agent.** The world is a small SQLite database, built fresh for each run, so a refund in one
test never leaks into the next. The rules are the agent under test: one order number; a refund for lost orders or
delivered-and-damaged ones; a lookup for "where is my parcel" questions; a ticket for everything else. Every tool call
is recorded, so the grader can check the path, not only the result.

The cells whose first line is `%%writefile` are a Jupyter magic. The magic saves the rest of the cell into the file
named after it, and the cell runs no Python. Run the `%%writefile` cells in order, top to bottom: the test file
imports `support_agent.py`, so that file must exist first.

In [2]:
%%writefile ../data/08/support_agent.py
import re
import sqlite3


def make_db(world: dict) -> sqlite3.Connection:
    """A fresh in-memory database for one run: the orders, and empty refund and ticket tables."""
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE orders (id TEXT PRIMARY KEY, status TEXT, total REAL)")
    db.executemany("INSERT INTO orders VALUES (?, ?, ?)", [(k, o["status"], o["total"]) for k, o in world["orders"].items()])
    db.execute("CREATE TABLE refunds (order_id TEXT, amount REAL)")
    db.execute("CREATE TABLE tickets (order_id TEXT)")
    return db


def run_support_agent(request: str, db: sqlite3.Connection) -> list[str]:
    """Scripted stand-in for the agent: fixed rules, no model. Writes to db and returns the tool names it called."""
    calls = []

    def call(name: str, order_id: str) -> None:
        calls.append(name)  # every call is recorded for the trajectory grader
        if name == "issue_refund":  # the refund tool takes only the order id: the amount comes from the order
            db.execute("INSERT INTO refunds SELECT id, total FROM orders WHERE id = ?", (order_id,))
        if name == "open_ticket":
            db.execute("INSERT INTO tickets VALUES (?)", (order_id,))

    text = request.lower()
    found = re.findall(r"\b(10\d\d)\b", request)  # order numbers: four digits that start with 10, such as 1003
    row = db.execute("SELECT status FROM orders WHERE id = ?", (found[0],)).fetchone() if found else None
    if row:  # returns [] and calls nothing if no known order is in the request
        order_id, status = found[0], row[0]
        if "refund" in text:
            if status == "lost" or (status == "delivered" and "damaged" in text):
                call("issue_refund", order_id)
            else:
                call("open_ticket", order_id)
        elif any(word in text for word in ("where", "late", "parcel", "status", "shipped", "arrive")):
            call("lookup_order", order_id)
        else:
            call("open_ticket", order_id)
    return calls

Overwriting ../data/08/support_agent.py


**Step 3 · the test file.** The library here is pytest, with plain `assert`s and `parametrize`. `run_and_check` runs
the agent on one task, reads the end state back with SQL, and returns the problems it finds. An empty list means the
task passed. A test per task reports each one; a gate counts the passes.

Each golden task can list `must_call` (tools that must run) and `forbidden_tools` (tools that must not run). Both are
optional. The test file imports the `google-genai` package (the model SDK), installed in this course's venv; the guard
below makes any attempt to create a model client fail. `monkeypatch` swaps a function for one test and restores it
after; `autouse=True` applies the fixture to every test without naming it.

In [3]:
%%writefile ../data/08/test_support_eval.py
import json
from pathlib import Path

import google.genai as genai  # the model SDK, imported only so the guard below can stop any model call
import pytest
from support_agent import make_db, run_support_agent

GOLDEN = json.loads((Path(__file__).resolve().parents[1] / "support_eval.json").read_text())
KNOWN_GAPS = {  # the misses of the first run; the course keeps them as the baseline: lesson 22 analyses why each one
    # fails, and lesson 28 keeps the gate in CI
    "t04": "'has not come yet' has no keyword, so the rules open a ticket",
    "t06": "'never came' and 'money back' are not in the rules",
    "t10": "'broken' is not in the damage rule",
    "t15": "two asks in one message; the rules see only the refund",
    "t16": "two orders; the rules handle only the first order number",
    "t24": "'smashed' and 'money back' are not in the rules",
    "t28": "two asks; the refund runs and the lookup is skipped",
}


@pytest.fixture(autouse=True)
def no_model_call(monkeypatch):
    # Guard: any attempt to create a Gemini client fails this test. The scripted agent never does.
    def refuse(self, *args, **kwargs):
        raise RuntimeError("this test must not call a model")
    monkeypatch.setattr(genai.Client, "__init__", refuse)


def cases() -> list:
    return [pytest.param(t, id=t["id"], marks=[pytest.mark.xfail(strict=True, reason=KNOWN_GAPS[t["id"]])]
                         if t["id"] in KNOWN_GAPS else []) for t in GOLDEN["tasks"]]

Overwriting ../data/08/test_support_eval.py


**The checks.** Outcome: each key in `expected_end` must equal the database's end state. Trajectory: no forbidden tool
ran, and every required tool did. The seven misses of the first run are marked `xfail` with a reason, so the suite
stays green and each miss stays visible. `strict=True` makes a fixed miss fail until its mark is removed.

In [4]:
%%writefile -a ../data/08/test_support_eval.py


def end_state(db) -> dict:
    """What the world looks like after a run, read back with SQL."""
    return {"refunds": {oid: amount for oid, amount in db.execute("SELECT order_id, amount FROM refunds")},
            "tickets": sorted(oid for (oid,) in db.execute("SELECT order_id FROM tickets"))}


def run_and_check(task: dict) -> list[str]:
    """Run the scripted agent on one task. Returns the problems found; an empty list means the task passed."""
    db = make_db(GOLDEN["world"])
    calls = set(run_support_agent(task["request"], db))
    end = end_state(db)
    problems = [f"{k}: expected {v}, got {end[k]}" for k, v in task["expected_end"].items() if end[k] != v]
    problems += [f"forbidden tool ran: {t}" for t in calls & set(task.get("forbidden_tools", []))]
    problems += [f"required tool did not run: {t}" for t in set(task.get("must_call", [])) - calls]
    return problems

Appending to ../data/08/test_support_eval.py


**Step 4 · the tests.** One test per golden task, and one gate on the success rate. The gate is the first baseline:
set it just under today's score, then raise it as the agent improves. Lesson 28 runs it in CI.

In [5]:
%%writefile -a ../data/08/test_support_eval.py


@pytest.mark.parametrize("task", cases())
def test_task(task):
    problems = run_and_check(task)
    assert not problems, "; ".join(problems)


def test_success_rate_gate():
    passed = sum(not run_and_check(t) for t in GOLDEN["tasks"])
    rate = passed / len(GOLDEN["tasks"])
    assert rate >= 0.75, f"task success rate {rate:.0%} is below the 75% gate"  # today's score is 77%: 75% sits just under it

Appending to ../data/08/test_support_eval.py


**Step 5 · run the suite.** The notebook runs pytest as a separate process, the same command you would type in a
terminal: `python -m pytest -q`. The last line is the summary. `xfail` counts as green; `failed` would be red.
The 31 tests are 30 tasks and the gate: 23 tasks pass, 7 are known misses (xfail), and the gate passes.

In [6]:
import subprocess

run = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "test_support_eval.py"],
                     cwd=RUN, capture_output=True, text=True)
print(run.stdout.strip().splitlines()[-1].rsplit(" in ", 1)[0])  # 24 passed (23 tasks and the gate), 7 xfailed
print("exit code:", run.returncode, "(0 means the suite is green)")

24 passed, 7 xfailed
exit code: 0 (0 means the suite is green)


Read the summary line: **24 passed, 7 xfailed**. That is 23 passing tasks plus the gate, and the 7 known misses.

**Step 6 · read the misses.** Run the same checks and print every task the agent misses. A task with a reason is a
known miss; a task without one would be a new regression.

In [7]:
sys.path.insert(0, str(RUN))
from test_support_eval import GOLDEN, KNOWN_GAPS, run_and_check  # noqa: E402

problems = {t["id"]: run_and_check(t) for t in GOLDEN["tasks"]}
passed = sum(not p for p in problems.values())
print(f"task success rate: {passed / len(problems):.0%} ({passed} of {len(problems)})")
for t in GOLDEN["tasks"]:
    if problems[t["id"]]:
        print(t["id"], "|", t["request"][:48], "|", KNOWN_GAPS.get(t["id"], "NEW miss"))

task success rate: 77% (23 of 30)
t04 | My order 1001 has not come yet. | 'has not come yet' has no keyword, so the rules open a ticket
t06 | Order 1011 never came, I want my money back. | 'never came' and 'money back' are not in the rules
t10 | Order 1004 arrived broken, refund please. | 'broken' is not in the damage rule
t15 | Where is order 1005? And refund it if it's late. | two asks in one message; the rules see only the refund
t16 | Refund orders 1003 and 1007, both were lost. | two orders; the rules handle only the first order number
t24 | The box for 1004 is smashed, I want my money bac | 'smashed' and 'money back' are not in the rules
t28 | Refund order 1012, it arrived damaged, and tell  | two asks; the refund runs and the lookup is skipped


## Under the hood
The library runs the tests; the numbers are plain counting. This section is the arithmetic behind the success rate and
the two repeat-run measures. **pass^k** is the chance that all `k` runs of a task succeed. **pass@k** is the chance that
at least one of the `k` runs succeeds. Both assume each run succeeds with the same chance `p` and the runs are
independent. With a real model, running a task k times gives different results: one run can pass and the next fail.
The example uses an assumed 80% rate, not a model result. It is a round number so the arithmetic is easy to follow.

In [8]:
def success_rate(passed: list[bool]) -> float:
    """Task success rate: passed tasks divided by all tasks."""
    return sum(passed) / len(passed)


def pass_hat_k(p: float, k: int) -> float:  # all k runs succeed
    return p ** k


def pass_at_k(p: float, k: int) -> float:  # at least one of k runs succeeds
    return 1 - (1 - p) ** k


print(f"success rate of this run: {success_rate([not p for p in problems.values()]):.0%}")
print(f"p = 0.8, k = 3: pass^3 = {pass_hat_k(0.8, 3):.1%}, pass@3 = {pass_at_k(0.8, 3):.1%}")

success rate of this run: 77%
p = 0.8, k = 3: pass^3 = 51.2%, pass@3 = 99.2%


## At work
- Teams start with 20 to 50 golden tasks, taken from real failures, and grade the end state the customer cares about,
  not the wording of the reply → lesson 22 (error analysis, and judges for free-text answers).
- A trajectory check catches the right answer reached by a forbidden path, such as a refund tool that should never run
  for that request → lesson 09 (least privilege and approval for writes).
- The same suite runs on every prompt, tool and model change, and becomes the release gate in CI → lesson 28. Evals
  appear in up to 66% of the agent job postings in our sample, the most-asked skill.

## Common mistakes
1. **A typo in a key of the golden set.** The grader reads the end state by the key you wrote. The key here is
   `refund`; the end state has `refunds`. The grader stops with a `KeyError`, which is what you want: a silent typo
   would pass every task.

This cell is meant to raise an error: it shows the KeyError you get from a typo in the golden set.

In [9]:
bad = {"id": "tX", "request": "Order 1003 is lost, please refund it.", "expected_end": {"refund": {"1003": 89.5}}}
print(run_and_check(bad))  # the end state's key is "refunds"

KeyError: 'refund'

2. **Grading the wording.** A check like `"refunded" in reply` fails when the model writes "I have issued your refund",
   and passes when the reply says "refunded" but no refund was made. Grade the end state; judge the wording later
   (lesson 22).
3. **A plain `xfail`, with no `strict`.** A miss that starts passing shows as a silent pass, and nobody removes the
   mark. Use `strict=True`, as Step 3 does.

## Try it
Write `task31` for the request "Order 1003 is lost, please refund it." Expect the refund of 89.5 for order 1003 and no
ticket, with `open_ticket` listed in `forbidden_tools`. Then run `run_and_check` on it and print the result.
Predict first whether the list of problems is empty.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
task31 = {"id": "t31", "request": "Order 1003 is lost, please refund it.",
          "expected_end": {"refunds": {"1003": 89.5}, "tickets": []}, "forbidden_tools": ["open_ticket"]}
print(run_and_check(task31) or "passed")  # passed: the rule refunds lost orders

passed


## Recap
- A golden set holds tasks with a known right answer: the request and the expected end state, not the wording of a reply.
- Outcome grading checks the world (the refund exists); trajectory grading checks the path (no forbidden tool ran).
  The task success rate is passed tasks over all tasks.
- One run proves little for a real model: pass^k is the chance all k runs pass, pass@k the chance at least one does.
- The eval is pytest with no model call: a scripted agent and a fresh SQLite world make it deterministic and free.

**Interview:** "How do you know an agent change made it better, and not just different?" Strong answer: "I run a golden
set of real tasks, grade the end state and the tool path, and compare the success rate before and after. Then I read
each new miss. A change that wins on average but adds one forbidden refund is still a failure."